# **Lab Test Data Cleaning**

## 1. Importing Libraries and Loading Data

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

import sys
sys.path.append(str(Path.cwd().parent))

from scripts.paths import RAW_DIR, CLEANED_DIR

df = pd.read_csv(RAW_DIR / "09_lab_test_master.csv")

In [2]:
df.head()

,Lab_Test_ID,Test_Name,Test_Category,Standard_Cost,Turnaround_Hours
0,T600000,MRI,Hormone,3222.39,24
1,T600001,Urine Routine,Hormone,6335.64,24
2,T600002,CT Scan,Hormone,1659.41,6
3,T600003,KFT,Blood,4181.51,72
4,T600004,CBC,Blood,3645.10,2


## 2. Initial Data Inspection

In [3]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())

Shape: (140, 5)

Columns:
['Lab_Test_ID', 'Test_Name', 'Test_Category', 'Standard_Cost', 'Turnaround_Hours']

Data types:
Lab_Test_ID          object
Test_Name            object
Test_Category        object
Standard_Cost       float64
Turnaround_Hours      int64
dtype: object

Missing values:
Lab_Test_ID         0
Test_Name           0
Test_Category       0
Standard_Cost       0
Turnaround_Hours    0
dtype: int64

Duplicate rows: 0


In [4]:
print("Unique values per column:")
for col in df.columns:
    print(f"\n{col}: {df[col].nunique(dropna=False)}")

Unique values per column:

Lab_Test_ID: 140

Test_Name: 14

Test_Category: 6

Standard_Cost: 140

Turnaround_Hours: 7


## 3. Audit of Specific Columns

In [5]:
# Lab_Test_ID audit
print("Lab_Test_ID duplicate count:", df["Lab_Test_ID"].duplicated().sum())
print("Lab_Test_ID missing count:", df["Lab_Test_ID"].isna().sum())
print("Lab_Test_ID invalid format count:", (
    df["Lab_Test_ID"].astype("string").str.fullmatch(r"T\d{6}", na=False).eq(False)
    & df["Lab_Test_ID"].notna()
).sum())

# Test_Name audit
print("\nTest_Name missing count:", df["Test_Name"].isna().sum())
print("Test_Name whitespace-only count:", (
    df["Test_Name"].astype("string").str.strip().eq("")
).sum())
print("Test_Name values:")
print(sorted(df["Test_Name"].dropna().unique()))

Lab_Test_ID duplicate count: 0
Lab_Test_ID missing count: 0
Lab_Test_ID invalid format count: 0

Test_Name missing count: 0
Test_Name whitespace-only count: 0
Test_Name values:
['Blood Sugar', 'CBC', 'CT Scan', 'ECG', 'HbA1c', 'KFT', 'LFT', 'Lipid Profile', 'MRI', 'TSH', 'Troponin', 'Ultrasound', 'Urine Routine', 'X-Ray']


In [6]:
# Test_Category audit
print("Test_Category missing count:", df["Test_Category"].isna().sum())
print("Test_Category whitespace-only count:", (
    df["Test_Category"].astype("string").str.strip().eq("")
).sum())
print("Test_Category values:")
print(sorted(df["Test_Category"].dropna().unique()))

# Review combinations without assuming that a category is wrong.
print("\nTest_Name/Test_Category combinations:")
print(
    df[["Test_Name", "Test_Category"]]
    .drop_duplicates()
    .sort_values(["Test_Name", "Test_Category"])
    .to_string(index=False)
)

Test_Category missing count: 0
Test_Category whitespace-only count: 0
Test_Category values:
['Blood', 'Cardiac', 'Hormone', 'Imaging', 'Other', 'Urine']

Test_Name/Test_Category combinations:
    Test_Name Test_Category
  Blood Sugar         Blood
  Blood Sugar       Cardiac
  Blood Sugar       Imaging
  Blood Sugar         Urine
          CBC         Blood
          CBC       Hormone
          CBC       Imaging
          CBC         Other
          CBC         Urine
      CT Scan         Blood
      CT Scan       Cardiac
      CT Scan       Hormone
      CT Scan         Other
          ECG         Blood
          ECG       Hormone
          ECG       Imaging
          ECG         Other
          ECG         Urine
        HbA1c       Cardiac
        HbA1c       Hormone
        HbA1c       Imaging
          KFT         Blood
          KFT       Cardiac
          KFT       Hormone
          KFT       Imaging
          KFT         Urine
          LFT         Blood
          LFT       Horm

In [7]:
# Standard_Cost audit
standard_cost_numeric = pd.to_numeric(df["Standard_Cost"], errors="coerce")
print("Standard_Cost missing/non-numeric count:", standard_cost_numeric.isna().sum())
print("Standard_Cost non-positive count:", (standard_cost_numeric.dropna() <= 0).sum())
print("Standard_Cost min:", standard_cost_numeric.min())
print("Standard_Cost max:", standard_cost_numeric.max())

Standard_Cost missing/non-numeric count: 0
Standard_Cost non-positive count: 0
Standard_Cost min: 645.96
Standard_Cost max: 14980.93


In [8]:
# Turnaround_Hours audit
turnaround_numeric = pd.to_numeric(df["Turnaround_Hours"], errors="coerce")
print("Turnaround_Hours missing/non-numeric count:", turnaround_numeric.isna().sum())
print("Turnaround_Hours non-positive count:", (turnaround_numeric.dropna() <= 0).sum())
print("Turnaround_Hours unique values:")
print(sorted(turnaround_numeric.dropna().unique()))

Turnaround_Hours missing/non-numeric count: 0
Turnaround_Hours non-positive count: 0
Turnaround_Hours unique values:
[2, 4, 6, 12, 24, 48, 72]


## 4. Data Cleaning

### 4.1 Lab_Test_ID

In [9]:
lab_test_id_cleaned = df["Lab_Test_ID"].astype("string").str.strip().str.upper()
df["Lab_Test_ID"] = lab_test_id_cleaned

### 4.2 Test_Name

In [10]:
# Strip surrounding whitespace only. Preserve source capitalization such as MRI, CBC, ECG, and HbA1c.
test_name_cleaned = df["Test_Name"].astype("string").str.strip()
df["Test_Name"] = test_name_cleaned

### 4.3 Test_Category

In [11]:
test_category_cleaned = df["Test_Category"].astype("string").str.strip()
df["Test_Category"] = test_category_cleaned

### 4.4 Standard_Cost

In [12]:
standard_cost_cleaned = pd.to_numeric(df["Standard_Cost"], errors="coerce")
standard_cost_cleaned = standard_cost_cleaned.mask(standard_cost_cleaned <= 0)
df["Standard_Cost"] = standard_cost_cleaned

### 4.5 Turnaround_Hours

In [13]:
turnaround_cleaned = pd.to_numeric(df["Turnaround_Hours"], errors="coerce")
turnaround_cleaned = turnaround_cleaned.mask(turnaround_cleaned <= 0)
df["Turnaround_Hours"] = turnaround_cleaned.astype("Int64")

## 5. Final Validation

In [14]:
print("Shape after cleaning:", df.shape)
print("\nMissing values:")
print(df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("Lab_Test_ID duplicates:", df["Lab_Test_ID"].duplicated().sum())
print("Lab_Test_ID invalid format:", (
    df["Lab_Test_ID"].notna()
    & ~df["Lab_Test_ID"].astype("string").str.fullmatch(r"T\d{6}", na=False)
).sum())
print("Non-positive Standard_Cost:", (df["Standard_Cost"].dropna() <= 0).sum())
print("Non-positive Turnaround_Hours:", (df["Turnaround_Hours"].dropna() <= 0).sum())
print("\nData types:")
print(df.dtypes)

Shape after cleaning: (140, 5)

Missing values:
Lab_Test_ID         0
Test_Name           0
Test_Category       0
Standard_Cost       0
Turnaround_Hours    0
dtype: int64

Duplicate rows: 0
Lab_Test_ID duplicates: 0
Lab_Test_ID invalid format: 0
Non-positive Standard_Cost: 0
Non-positive Turnaround_Hours: 0

Data types:
Lab_Test_ID         string[python]
Test_Name           string[python]
Test_Category       string[python]
Standard_Cost              float64
Turnaround_Hours             Int64
dtype: object


In [15]:
print("Final Test_Name values:")
print(sorted(df["Test_Name"].dropna().unique()))

print("\nFinal Test_Category values:")
print(sorted(df["Test_Category"].dropna().unique()))

print("\nFinal Standard_Cost summary:")
print(df["Standard_Cost"].describe())

print("\nFinal Turnaround_Hours values:")
print(sorted(df["Turnaround_Hours"].dropna().unique()))

Final Test_Name values:
['Blood Sugar', 'CBC', 'CT Scan', 'ECG', 'HbA1c', 'KFT', 'LFT', 'Lipid Profile', 'MRI', 'TSH', 'Troponin', 'Ultrasound', 'Urine Routine', 'X-Ray']

Final Test_Category values:
['Blood', 'Cardiac', 'Hormone', 'Imaging', 'Other', 'Urine']

Final Standard_Cost summary:
count      140.000000
mean      7375.581357
std       4237.996734
min        645.960000
25%       3607.737500
50%       7583.895000
75%      10704.575000
max      14980.930000
Name: Standard_Cost, dtype: float64

Final Turnaround_Hours values:
[2, 4, 6, 12, 24, 48, 72]


## 6. Export Cleaned Data

In [17]:
df.to_csv(CLEANED_DIR / "09_lab_tests_cleaned.csv", index=False)